# Setup del entorno en Kaggle / Colab (T02)

1. Activa la GPU (y en Kaggle, el Internet):
   - **Kaggle:** panel *Session options* → *Accelerator* `GPU T4 x2` o `GPU P100`, e *Internet* activado.
   - **Colab:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*.
2. Ejecuta todas las celdas (*Run All* en Kaggle, *Ejecutar todas* en Colab). Hay que hacerlo al empezar cada sesión.
3. La última celda debe terminar con `OK: el entorno coincide con las versiones fijadas.`

Si algo falla, consulta la sección *Si algo falla* de `docs/reproducibilidad/entorno.md`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

BRANCH = "main"  # cambiar para probar la rama de un PR
REPO_URL = "https://github.com/Ronald1301/Lymphoma-Machine-Learning.git"
UV_VERSION = "0.13.8"  # la versión de uv con la que se generó uv.lock

ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
ON_COLAB = "COLAB_RELEASE_TAG" in os.environ
if not (ON_KAGGLE or ON_COLAB):
    raise RuntimeError("Este notebook es solo para Kaggle o Colab. En la laptop usa los comandos de docs/reproducibilidad/entorno.md.")
ROOT = Path("/kaggle/working") if ON_KAGGLE else Path("/content")
REPO_DIR = ROOT / "Lymphoma-Machine-Learning"

os.environ["UV_PROJECT_ENVIRONMENT"] = "/tmp/lymphoma-venv"  # fuera de /kaggle/working: no se guarda como output
os.environ["UV_CACHE_DIR"] = "/tmp/uv-cache"
os.environ["UV_FROZEN"] = "1"  # instala exactamente lo que dice uv.lock, sin volver a resolver

UV = [sys.executable, "-m", "uv"]


def run(*cmd: str) -> None:
    """Ejecuta un comando mostrando su salida; lanza un error si falla."""
    print("$", " ".join(cmd), flush=True)
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
    if proc.returncode != 0:
        raise RuntimeError(f"Falló (código {proc.returncode}): {' '.join(cmd)}")


print("Plataforma:", "Kaggle" if ON_KAGGLE else "Colab", "| repo:", REPO_DIR)

In [ ]:
run(sys.executable, "-m", "pip", "install", "-q", f"uv=={UV_VERSION}")
if REPO_DIR.exists():
    run("git", "-C", str(REPO_DIR), "pull", "--ff-only")
else:
    run("git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR))
os.chdir(REPO_DIR)
run("git", "log", "-1", "--oneline")

In [ ]:
# Descarga Python 3.12 y torch cu126 (T4 y P100); unos minutos la primera vez en cada sesión.
run(*UV, "sync")

In [ ]:
run(*UV, "run", "python", "-c",
    "import monai, torch; print('monai', monai.__version__, '| torch', torch.__version__, '| CUDA', torch.cuda.is_available())")
run(*UV, "run", "python", "scripts/check_env.py", "--save")

## Siguientes pasos

- **Evidencia (solo para T02):** descarga `Lymphoma-Machine-Learning/results/env/env_kaggle.json` (panel *Output* en Kaggle) o `env_colab.json` (panel *Archivos* en Colab) y súbelo al repo en `results/env/`.
- **Para entrenar:** copia las celdas de código de este notebook al principio del tuyo y lanza el script con `run(*UV, "run", "python", "scripts/train.py", "--config", "configs/<experimento>.yaml")`.